In [ ]:
import pandas as pd
import glob
import os

print("SENTRi-X Environment Active.")
print(f"Pandas Version: {pd.__version__}")

# 1. Define the relative path from the 'notebooks' folder to the ToN-IoT folder
ton_iot_path = "../data/raw/ton_iot/"

# 2. Use glob to automatically find all 23 CSV files in that folder
all_files = glob.glob(os.path.join(ton_iot_path, "*.csv"))
print(f"\nSuccessfully located {len(all_files)} ToN-IoT chunk files.")

# 3. Load ONLY the first chunk to prototype our ETL Pipeline safely
print(f"Loading {os.path.basename(all_files[0])} into Pandas...")
df = pd.read_csv(all_files[0])

print("\nExtraction Complete! Here is the shape of this single chunk:")
print(f"Total Rows (Network Flows): {df.shape[0]:,}")
print(f"Total Columns (Features): {df.shape[1]}")

# 4. Display the first 5 rows to verify the column headers
df.head()

In [ ]:
import numpy as np

print("--- Step 2: Data Cleaning & Feature Selection ---")

# 1. Fix the DtypeWarning (Mixed Types)
# We force Pandas to turn 'src_bytes' (and other math columns) into strict numbers. 
# The 'coerce' command tells it: "If you see a dash or a letter, turn it into a blank NaN."
numeric_columns = ['src_bytes', 'dst_bytes'] 
for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
print("Successfully coerced mixed data types into strict numeric formats.")

# 2. Drop Overfitting Features
# We drop IPs and Timestamps so the AI learns the behavior of the attack, not the location.
drop_cols = ['ts', 'src_ip', 'src_port', 'dst_ip', 'dst_port']
df.drop(columns=[col for col in drop_cols if col in df.columns], inplace=True)
print(f"Dropped contextual metadata. Features remaining: {df.shape[1]}")

# 3. Clean Missing and Infinite Values
# Replace any remaining string dashes '-' with mathematical NaN
df.replace('-', np.nan, inplace=True)
# Replace infinite network flow values with mathematical NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate how much data we drop
initial_rows = df.shape[0]
df.dropna(inplace=True)
final_rows = df.shape[0]

print(f"Purged {initial_rows - final_rows:,} corrupted rows (NaN/Inf).")
print(f"Cleaned Dataset Size: {final_rows:,} Rows")

df.head()

In [ ]:
import pandas as pd
import numpy as np

print("--- Step 2: Advanced Data Cleaning (Thresholding) ---")

# 1. We must reload the dataframe since the last script emptied it!
print("Reloading the dataset chunk...")
df = pd.read_csv(all_files[0], low_memory=False)

# 2. Fix the mixed data types
numeric_columns = ['src_bytes', 'dst_bytes'] 
for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 3. Drop Overfitting Features (IPs, Timestamps)
drop_cols = ['ts', 'src_ip', 'src_port', 'dst_ip', 'dst_port']
df.drop(columns=[col for col in drop_cols if col in df.columns], inplace=True)

# 4. Standardize missing values
df.replace('-', np.nan, inplace=True)
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# 5. Drop columns that are mostly empty BEFORE dropping rows
print("\nScanning for highly corrupted columns...")
threshold = 0.30 # If a column is missing more than 30% of its data, drop the column.
missing_percentages = df.isna().mean()
bad_columns = missing_percentages[missing_percentages > threshold].index.tolist()

if bad_columns:
    print(f"Dropping {len(bad_columns)} columns with >30% missing data:")
    print(bad_columns)
    df.drop(columns=bad_columns, inplace=True)
else:
    print("No columns exceeded the 30% missing data threshold.")

# 6. Now drop the remaining rows that have occasional NaNs
initial_rows = df.shape[0]
df.dropna(inplace=True)
final_rows = df.shape[0]

print(f"\nPurged {initial_rows - final_rows:,} corrupted rows.")
print(f"Cleaned Dataset Size: {final_rows:,} Rows")
print(f"Final Feature Count: {df.shape[1]} Columns")

df.head()

In [ ]:
print("--- Step 3: Feature Inspection & Target Identification ---")

# 1. Display the surviving 18 columns and their data types
print("Surviving Features and Data Types:")
print(df.dtypes)

# 2. Check the distribution of the target variables
print("\n--- Threat Distribution ---")
if 'label' in df.columns:
    print("Binary Target ('label'):")
    # Multiplying by 100 gives us the exact percentage split
    print((df['label'].value_counts(normalize=True) * 100).round(2).astype(str) + '%')

if 'type' in df.columns:
    print("\nMulticlass Target ('type'):")
    print(df['type'].value_counts())
else:
    print("\nColumn 'type' not found. We will predict purely on binary 'label'.")

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

print("--- Step 4: Feature Engineering & Scaling ---")

# 1. Isolate Features (X) and Target (y)
# We drop 'type' so the AI doesn't cheat by looking at the attack name.
X = df.drop(columns=['label', 'type'])
y = df['label'].astype(int)

print(f"Starting Feature Count: {X.shape[1]}")

# 2. One-Hot Encoding for categorical text
categorical_cols = ['proto', 'conn_state']
X = pd.get_dummies(X, columns=[col for col in categorical_cols if col in X.columns], drop_first=True)

# Convert all boolean columns generated by get_dummies to integers (0 or 1)
X = X.astype(float)

print(f"Feature Count after One-Hot Encoding: {X.shape[1]}")

# 3. Standardization (Scaling)
scaler = StandardScaler()

# Fit and transform the mathematical features
X_scaled = scaler.fit_transform(X)

# Convert back to a Pandas DataFrame for easy viewing
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print("\nTransformation Complete! The data is now a pure, normalized mathematical matrix.")
X_scaled_df.head()

In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

print("--- Step 5: Data Splitting & SMOTE Balancing ---")

# 1. Split the data FIRST (80% Training, 20% Testing)
# 'stratify=y' ensures the 79/21 ratio is perfectly maintained in both splits before we alter it.
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_df, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Original Training Set: {X_train.shape[0]:,} rows")
print(f"Original Testing Set:  {X_test.shape[0]:,} rows")

print("\n--- Before SMOTE (Training Set Imbalance) ---")
print((y_train.value_counts(normalize=True) * 100).round(2).astype(str) + '%')

# 2. Apply SMOTE to the training set ONLY
print("\nApplying SMOTE... (Injecting synthetic minority data)")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# 3. Verify the new, perfectly balanced training environment
print("\n--- After SMOTE (New Training Set Balance) ---")
print((y_train_smote.value_counts(normalize=True) * 100).round(2).astype(str) + '%')
print(f"New Training Set Size: {X_train_smote.shape[0]:,} rows")

In [ ]:
import joblib
import os

print("--- Step 6: Exporting the Processed ML Matrices (ToN-IoT) ---")

processed_dir = "../data/processed/"
os.makedirs(processed_dir, exist_ok=True)

# Appending 'ton_iot_' to the filenames to protect them from being overwritten later
print("Exporting ToN-IoT matrices to disk...")

joblib.dump(X_train_smote, os.path.join(processed_dir, "ton_iot_X_train_smote.pkl"))
joblib.dump(y_train_smote, os.path.join(processed_dir, "ton_iot_y_train_smote.pkl"))
joblib.dump(X_test, os.path.join(processed_dir, "ton_iot_X_test.pkl"))
joblib.dump(y_test, os.path.join(processed_dir, "ton_iot_y_test.pkl"))

joblib.dump(scaler, os.path.join(processed_dir, "ton_iot_scaler.pkl"))

print(f"\nPipeline Complete! ToN-IoT artifacts saved securely to: {processed_dir}")